# ChromaDB

In [1]:
documents = [
    "Python is a programming language.",
    "Python is commonly used for machine learning and data science.",
    "Vector databases store numerical representations of text called embeddings.",
    "Embeddings represent the meaning of text as numbers.",
    "Chroma and FAISS can be used for similarity search."
]

Create embeddings

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)


Create Chroma

In [3]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

In [4]:
collection = client.get_or_create_collection(
    name="documents",
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

Metadata

In [5]:
metadatas = [
    {
        "category": "programming",
        "source": "python.txt"
    },
    {
        "category": "machine_learning",
        "source": "ml.txt"
    },
    {
        "category": "vector_database",
        "source": "vector_db.txt"
    },
    {
        "category": "embedding",
        "source": "embeddings.txt"
    },
    {
        "category": "vector_search",
        "source": "search.txt"
    }
]

Store embeddings in Chroma

In [6]:
collection.add(
    ids=[str(i) for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Embeddings stored in ChromaDB")
print("Count:", collection.count())

Embeddings stored in ChromaDB
Count: 5


In [7]:
data = collection.get()
print(data)

{'ids': ['0', '1', '2', '3', '4'], 'embeddings': None, 'documents': ['Python is a programming language.', 'Python is commonly used for machine learning and data science.', 'Vector databases store numerical representations of text called embeddings.', 'Embeddings represent the meaning of text as numbers.', 'Chroma and FAISS can be used for similarity search.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'python.txt', 'category': 'programming'}, {'category': 'machine_learning', 'source': 'ml.txt'}, {'category': 'vector_database', 'source': 'vector_db.txt'}, {'category': 'embedding', 'source': 'embeddings.txt'}, {'category': 'vector_search', 'source': 'search.txt'}]}


User Query

In [8]:
query = "What is Python used for?"
query_embedding = model.encode([query])

## Chroma Search

In [9]:
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include=["documents", "distances", "metadatas"]
)

for doc in results["documents"][0]:
    print(doc)

Python is a programming language.
Python is commonly used for machine learning and data science.
Vector databases store numerical representations of text called embeddings.


Result

In [10]:
print(results)

{'ids': [['0', '1', '2']], 'embeddings': None, 'documents': [['Python is a programming language.', 'Python is commonly used for machine learning and data science.', 'Vector databases store numerical representations of text called embeddings.']], 'uris': None, 'included': ['documents', 'distances', 'metadatas'], 'data': None, 'metadatas': [[{'source': 'python.txt', 'category': 'programming'}, {'source': 'ml.txt', 'category': 'machine_learning'}, {'source': 'vector_db.txt', 'category': 'vector_database'}]], 'distances': [[0.1563226580619812, 0.19890785217285156, 0.8080843687057495]]}


In [11]:
for document, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):

    print(f"Distance : {distance:.4f}")
    print(f"Document : {document}")
    print(f"Metadata : {metadata}")
    print("-" * 50)

Distance : 0.1563
Document : Python is a programming language.
Metadata : {'source': 'python.txt', 'category': 'programming'}
--------------------------------------------------
Distance : 0.1989
Document : Python is commonly used for machine learning and data science.
Metadata : {'source': 'ml.txt', 'category': 'machine_learning'}
--------------------------------------------------
Distance : 0.8081
Document : Vector databases store numerical representations of text called embeddings.
Metadata : {'source': 'vector_db.txt', 'category': 'vector_database'}
--------------------------------------------------


In [12]:
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    where={
        "category": "machine_learning"
    },
    include=[
        "documents",
        "distances",
        "metadatas"
    ]
)

In [19]:
for document, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):

    print(f"Distance : {distance:.4f}")
    print(f"Document : {document}")
    print(f"Metadata : {metadata}")
    print("-" * 50)

Distance : 0.1989
Document : Python is commonly used for machine learning and data science.
Metadata : {'category': 'machine_learning', 'source': 'ml.txt'}
--------------------------------------------------


## Rerankeing

In [23]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [24]:
documents_found = results["documents"][0]
metadata_found = results["metadatas"][0]

In [25]:
candidates = list(
    zip(documents_found, metadata_found)
)

Create query-document pairs

In [26]:
pairs = [
    [query, document]
    for document, metadata in candidates
]

In [27]:
scores = reranker.predict(pairs)

In [28]:
ranked = sorted(
    zip(candidates, scores),
    key=lambda x: x[1],
    reverse=True
)

In [29]:
for rank, (document, score) in enumerate(
    ranked,
    start=1
):
    print(f"Rank: {rank}")
    print(f"Score: {score:.4f}")
    print(f"Document: {document}")
    print("-" * 50)

Rank: 1
Score: 9.8594
Document: ('Python is commonly used for machine learning and data science.', {'category': 'machine_learning', 'source': 'ml.txt'})
--------------------------------------------------


Take the top 2

In [30]:
top_results = ranked[:2]

In [31]:
for rank, ((document, metadata), score) in enumerate(
    top_results,
    start=1
):
    print(f"Rank: {rank}")
    print(f"Score: {score:.4f}")
    print(f"Document: {document}")
    print(f"Metadata: {metadata}")
    print("-" * 50)

Rank: 1
Score: 9.8594
Document: Python is commonly used for machine learning and data science.
Metadata: {'category': 'machine_learning', 'source': 'ml.txt'}
--------------------------------------------------


Build the context

In [32]:
context_parts = []

for (document, metadata), score in top_results:

    context_parts.append(
        f"""
Source: {metadata['source']}
Category: {metadata['category']}

Content:
{document}
"""
    )

context = "\n\n".join(context_parts)

print(context)


Source: ml.txt
Category: machine_learning

Content:
Python is commonly used for machine learning and data science.

